# Laboratory Exercise: SSD vs. YOLO — Object Detection Practical

| Field | Details |
|---|---|
| **Name** | Nimesh Timalsina |
| **Roll No** | 26 |
| **Date** | April 30, 2026 |

---

## Assignment Overview

In this practical, we act as **Model Auditors** investigating how two fundamentally different object detection philosophies handle challenging real-world visual scenarios:

| Philosophy | Representative Model | Core Approach |
|------------|---------------------|---------------|
| **Multi-scale feature map detection** | SSD300 (VGG16 backbone) | Predict at multiple spatial resolutions simultaneously |
| **Unified regression-based detection** | YOLOv8n / YOLOv11n | Single-pass direct regression from image pixels to boxes |

Rather than focusing purely on accuracy numbers, this lab emphasizes:
- **Architectural reasoning** — understanding *why* each model behaves differently
- **Empirical observation** — recording actual detection counts, latency, and box quality
- **Detection behavior analysis** — studying failure modes (missed small objects, crowded scenes)
- **Deployment-oriented decision making** — choosing the right model for a given use-case

## Lab Workflow

| Step | Description |
|------|-------------|
| 1 | Install dependencies and configure device |
| 2 | Prepare the 10-image stress-test dataset |
| 3 | Load pretrained SSD300 and YOLOv8n models |
| 4 | Run inference with fixed conf=0.25, IoU=0.45, imgsz=640 |
| 5 | Generate side-by-side annotated comparison images |
| 6 | Build the evaluation matrix (small objects, speed, localization) |
| 7 | Answer the three architectural deep-dive questions |
| 8 | Write the 200-word technical synthesis |

In [ ]:
# Install all required packages
!pip install -q ultralytics opencv-python torch torchvision matplotlib pandas Pillow

## Section 1 — Technical Requirements

### Software
- **Python 3.8+**
- `ultralytics` — YOLOv8/v11 training and inference
- `opencv-python` — image loading, drawing, and manipulation
- `torch` + `torchvision` — SSD model and tensor operations
- `matplotlib` + `pandas` — visualisation and evaluation matrix

### Models Used
| Model | Source | Pretrained On | Parameters |
|-------|--------|--------------|------------|
| **SSD300-VGG16** | `torchvision.models.detection` | COCO 2017 | ~26M |
| **YOLOv8n** | `ultralytics` | COCO 2017 | ~3M |

### Hardware
- GPU strongly recommended for SSD (large backbone)
- YOLOv8n runs comfortably on CPU
- MPS (Apple Silicon) supported via `torch.backends.mps`

### Inference Configuration (Fixed for Both Models)
| Parameter | Value |
|-----------|-------|
| Confidence Threshold | 0.25 |
| IoU Threshold | 0.45 |
| Input Resolution | 640 × 640 |
| Weights | Pretrained only — no fine-tuning |

In [ ]:
import torch
import torchvision
import torchvision.transforms.functional as TF
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import time
import urllib.request
from pathlib import Path
from PIL import Image, ImageDraw

def get_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

DEVICE = get_device()
print(f'PyTorch      : {torch.__version__}')
print(f'Torchvision  : {torchvision.__version__}')
print(f'Device       : {DEVICE}')
print(f'CUDA         : {torch.cuda.is_available()}')
print(f'MPS          : {torch.backends.mps.is_available()}')

## Section 2 — Dataset Preparation

### Required Image Categories

The evaluation set must contain **at least 10 images** across four stress-test categories:

| Category | Min. Images | What to Look For |
|----------|------------|------------------|
| `small_distant/` | 3 | Birds in sky, distant cars, aerial drone shots, boats at sea |
| `occluded_overlapping/` | 3 | People walking in doorways, stacked boxes, parked cars side-by-side |
| `crowded_scenes/` | 2 | Markets, stadiums, street intersections, pedestrian crossings |
| `unusual_aspect_ratio/` | 2 | Tall buildings, panorama landscapes, long vehicles, flagpoles |

### Image Sources
Images may be collected from:
- Personal photographs
- Open-source datasets (COCO, Open Images, Pascal VOC)
- Internet images (ensure license permits academic use)
- Publicly available benchmark repositories

### Directory Structure
```
stress_test_dataset/
├── small_distant/          ← 3+ images
├── occluded_overlapping/   ← 3+ images
├── crowded_scenes/         ← 2+ images
└── unusual_aspect_ratio/   ← 2+ images
```

> **Note:** The cell below creates the directory structure and generates synthetic placeholder images so the pipeline runs immediately. **Replace the placeholder images with real photographs** for meaningful results.

In [ ]:
DATASET_DIR = Path('stress_test_dataset')

CATEGORIES = {
    'small_distant': 3,
    'occluded_overlapping': 3,
    'crowded_scenes': 2,
    'unusual_aspect_ratio': 2,
}

for cat in CATEGORIES:
    (DATASET_DIR / cat).mkdir(parents=True, exist_ok=True)

def make_placeholder(path: Path, width: int, height: int, scene: str):
    """Generate a synthetic placeholder image if the path does not exist."""
    if path.exists():
        return
    img = Image.new('RGB', (width, height), (180, 200, 220))
    draw = ImageDraw.Draw(img)
    rng = np.random.default_rng(hash(scene) % 2**32)

    if scene == 'small_distant':
        for _ in range(12):
            x, y = rng.integers(10, width-10), rng.integers(10, height-10)
            r = rng.integers(3, 10)
            c = tuple(rng.integers(50, 200, 3).tolist())
            draw.ellipse([x-r, y-r, x+r, y+r], fill=c)
    elif scene == 'occluded_overlapping':
        for _ in range(6):
            x1, y1 = rng.integers(0, width//2), rng.integers(0, height//2)
            bw, bh = rng.integers(80, 200), rng.integers(100, 250)
            c = tuple(rng.integers(80, 220, 3).tolist())
            draw.rectangle([x1, y1, x1+bw, y1+bh], fill=c, outline=(30,30,30), width=2)
    elif scene == 'crowded_scenes':
        for _ in range(30):
            x, y = rng.integers(0, width-20), rng.integers(0, height-20)
            w, h = rng.integers(15, 40), rng.integers(20, 60)
            c = tuple(rng.integers(60, 200, 3).tolist())
            draw.rectangle([x, y, x+w, y+h], fill=c)
    elif scene == 'unusual_aspect_ratio':
        for _ in range(4):
            x1 = rng.integers(0, width-30)
            bh = rng.integers(height//2, height)
            c = tuple(rng.integers(80, 200, 3).tolist())
            draw.rectangle([x1, 0, x1+25, bh], fill=c)

    img.save(path)

placeholders = [
    ('small_distant',         'bird_sky_1.jpg',     640, 480,  'small_distant'),
    ('small_distant',         'distant_cars_1.jpg',  640, 480,  'small_distant'),
    ('small_distant',         'aerial_view_1.jpg',   640, 480,  'small_distant'),
    ('occluded_overlapping',  'overlapping_1.jpg',   640, 640,  'occluded_overlapping'),
    ('occluded_overlapping',  'occluded_2.jpg',      640, 640,  'occluded_overlapping'),
    ('occluded_overlapping',  'shelf_items_3.jpg',   640, 640,  'occluded_overlapping'),
    ('crowded_scenes',        'market_crowd_1.jpg',  640, 480,  'crowded_scenes'),
    ('crowded_scenes',        'stadium_1.jpg',       640, 480,  'crowded_scenes'),
    ('unusual_aspect_ratio',  'tall_building_1.jpg', 320, 960,  'unusual_aspect_ratio'),
    ('unusual_aspect_ratio',  'panorama_1.jpg',      1280, 320, 'unusual_aspect_ratio'),
]

for cat, fname, w, h, scene in placeholders:
    make_placeholder(DATASET_DIR / cat / fname, w, h, scene)

all_images = sorted(DATASET_DIR.rglob('*.jpg')) + sorted(DATASET_DIR.rglob('*.png'))
print(f'Total images in dataset: {len(all_images)}')
for p in all_images:
    print(f'  [{p.parent.name}] {p.name}')

## Section 3 — Experimental Configuration

All inference runs use the following fixed settings (as specified in the assignment):

| Parameter | Value | Rationale |
|-----------|-------|----------|
| `CONF_THRESH` | 0.25 | Relatively permissive — catches weak detections |
| `IOU_THRESH` | 0.45 | Slightly aggressive NMS — reduces duplicate boxes |
| `IMG_SIZE` | 640 × 640 | Square input; YOLOv8 standard; SSD resizes to 300×300 internally |
| Pretrained weights | COCO 2017 | Both models trained on the same 80-class dataset |

> **Neither model is fine-tuned.** This ensures the comparison reflects architecture, not training regimen.

In [ ]:
CONF_THRESH  = 0.25
IOU_THRESH   = 0.45
IMG_SIZE     = 640

# COCO 80-class names (YOLOv8 0-indexed)
YOLO_CLASSES = [
    'person','bicycle','car','motorcycle','airplane','bus','train','truck','boat',
    'traffic light','fire hydrant','stop sign','parking meter','bench','bird','cat',
    'dog','horse','sheep','cow','elephant','bear','zebra','giraffe','backpack',
    'umbrella','handbag','tie','suitcase','frisbee','skis','snowboard','sports ball',
    'kite','baseball bat','baseball glove','skateboard','surfboard','tennis racket',
    'bottle','wine glass','cup','fork','knife','spoon','bowl','banana','apple',
    'sandwich','orange','broccoli','carrot','hot dog','pizza','donut','cake',
    'chair','couch','potted plant','bed','dining table','toilet','tv','laptop',
    'mouse','remote','keyboard','cell phone','microwave','oven','toaster','sink',
    'refrigerator','book','clock','vase','scissors','teddy bear','hair drier','toothbrush'
]

# torchvision SSD uses COCO category IDs (1-indexed, gaps at 12,26,29,30,...)
# weights.meta['categories'] gives the correct list in order
SSD_CLASSES_ORDERED = None  # filled after loading model

print(f'Config: conf={CONF_THRESH}, iou={IOU_THRESH}, img_size={IMG_SIZE}')
print(f'YOLOv8 classes: {len(YOLO_CLASSES)}')

## Section 4 — Load Models

### SSD300 — Single Shot MultiBox Detector

**Architecture Summary:**
```
Input Image (resized to 300×300 internally)
        │
   VGG-16 Backbone (Conv1–Conv5, truncated)
        │
   Extra feature extraction layers (Conv6–Conv11)
        │
   ┌────┴────────────────────────────────┐
   │ 6 Detection Heads at different scales:
   │  Conv4_3  FC7    Conv8_2  Conv9_2  Conv10_2  Conv11_2
   │  38×38   19×19   10×10    5×5       3×3       1×1
   └────────────────────────────────────┘
        │
   8732 default anchor boxes → NMS → final detections
```

**Key design decisions:**
- Multi-scale feature maps allow detecting objects at different sizes on different feature levels.
- Default boxes (anchors) at each location remove the need for a separate proposal stage.
- Hard negative mining (3:1 ratio) compensates for anchor imbalance.

**Torchvision implementation:** `ssd300_vgg16(weights=SSD300_VGG16_Weights.DEFAULT)` — COCO pretrained.

In [ ]:
from torchvision.models.detection import ssd300_vgg16, SSD300_VGG16_Weights

ssd_weights = SSD300_VGG16_Weights.DEFAULT
ssd_model = ssd300_vgg16(weights=ssd_weights)
ssd_model.eval()
ssd_model = ssd_model.to(DEVICE)

# Extract ordered class name list from weights metadata
SSD_CLASSES_ORDERED = ssd_weights.meta['categories']

ssd_params = sum(p.numel() for p in ssd_model.parameters())
print(f'SSD300-VGG16 loaded   — {ssd_params:,} parameters')
print(f'SSD classes           : {len(SSD_CLASSES_ORDERED)} (including background)')

### YOLOv8n — You Only Look Once (v8, Nano variant)

**Architecture Summary:**
```
Input Image (640×640, letterboxed)
        │
   CSPDarknet Backbone (C2f modules)
   Down-samples: /2 → /4 → /8 → /16 → /32
        │
   PAN-FPN Neck (Path Aggregation Network)
   Up-samples and merges feature scales
        │
   Decoupled Detection Head (3 output scales)
   — Separate classification and regression branches
   — Anchor-FREE: predicts object centers directly
        │
   TaskAligned Assigner (training) → NMS (inference)
        │
   Final detections: [x_center, y_center, w, h, conf, class]
```

**Key design decisions vs SSD:**
- **No anchors** — eliminates anchor design hyperparameters and anchor-object mismatch.
- **Decoupled head** — classification and box regression are separate branches, avoiding task interference.
- **Single inference pass** — no proposal generation stage → lower latency.
- **CIoU loss** — penalizes aspect ratio, center distance, and overlap simultaneously.

In [ ]:
from ultralytics import YOLO

yolo_model = YOLO('yolov8n.pt')   # downloads ~6MB on first run

yolo_params = sum(p.numel() for p in yolo_model.model.parameters())
print(f'YOLOv8n loaded        — {yolo_params:,} parameters')
print(f'YOLO classes          : {len(YOLO_CLASSES)}')

# Architecture comparison table
import pandas as pd
arch_df = pd.DataFrame([
    {'Property': 'Backbone',            'SSD300': 'VGG-16',               'YOLOv8n': 'CSPDarknet (C2f)'},
    {'Property': 'Neck',                'SSD300': 'Extra conv layers',     'YOLOv8n': 'PAN-FPN'},
    {'Property': 'Head',                'SSD300': 'Coupled (cls+reg)',      'YOLOv8n': 'Decoupled (cls | reg)'},
    {'Property': 'Anchor strategy',     'SSD300': 'Anchor-based (8732)',    'YOLOv8n': 'Anchor-free'},
    {'Property': 'Detection scales',    'SSD300': '6 feature map levels',   'YOLOv8n': '3 feature map levels'},
    {'Property': 'Input resolution',    'SSD300': '300×300',               'YOLOv8n': '640×640'},
    {'Property': 'Parameters',          'SSD300': f'~{ssd_params//1_000_000}M', 'YOLOv8n': f'~{yolo_params//1_000_000}M'},
    {'Property': 'Pretrained on',       'SSD300': 'COCO 2017',             'YOLOv8n': 'COCO 2017'},
])
print(arch_df.to_string(index=False))

## Section 5 — Inference Pipeline

### SSD Inference Notes
- Input: PIL image converted to float tensor [0,1] via `TF.to_tensor()`.
- SSD internally resizes to 300×300 — **the 640 config does not apply to SSD's internal resolution**.
- Output: dict with `boxes` [N,4] (xyxy), `labels` [N], `scores` [N].
- Latency measured with `time.perf_counter()` around the model forward pass.

### YOLOv8 Inference Notes
- Ultralytics handles all preprocessing (letterboxing to 640×640).
- `imgsz=640` passed explicitly to match assignment config.
- Latency measured around the full `model()` call (includes pre/postprocessing).
- `verbose=False` suppresses per-image Ultralytics output.

In [ ]:
def run_ssd(image_path, model, device, conf_thresh=0.25):
    """Run SSD300 inference on a single image. Returns detection dict."""
    img_pil = Image.open(image_path).convert('RGB')
    img_tensor = TF.to_tensor(img_pil).to(device)   # [C, H, W] in [0,1]

    t0 = time.perf_counter()
    with torch.no_grad():
        outputs = model([img_tensor])
    latency_ms = (time.perf_counter() - t0) * 1000

    out = outputs[0]
    keep = out['scores'] >= conf_thresh
    boxes  = out['boxes'][keep].cpu().numpy()    # xyxy pixel coords
    labels = out['labels'][keep].cpu().numpy()
    scores = out['scores'][keep].cpu().numpy()

    return {
        'boxes': boxes,
        'labels': labels,
        'scores': scores,
        'n_det': int(keep.sum()),
        'latency_ms': latency_ms,
        'image': np.array(img_pil),
        'category': Path(image_path).parent.name,
        'filename': Path(image_path).name,
    }

In [ ]:
def run_yolo(image_path, model, conf_thresh=0.25, iou_thresh=0.45, img_size=640):
    """Run YOLOv8 inference on a single image. Returns detection dict."""
    t0 = time.perf_counter()
    results = model(
        str(image_path),
        conf=conf_thresh,
        iou=iou_thresh,
        imgsz=img_size,
        verbose=False,
    )
    latency_ms = (time.perf_counter() - t0) * 1000

    r = results[0]
    if r.boxes and len(r.boxes) > 0:
        boxes  = r.boxes.xyxy.cpu().numpy()
        labels = r.boxes.cls.cpu().numpy().astype(int)
        scores = r.boxes.conf.cpu().numpy()
    else:
        boxes  = np.zeros((0, 4))
        labels = np.zeros(0, dtype=int)
        scores = np.zeros(0)

    img_pil = Image.open(image_path).convert('RGB')
    return {
        'boxes': boxes,
        'labels': labels,
        'scores': scores,
        'n_det': len(boxes),
        'latency_ms': latency_ms,
        'image': np.array(img_pil),
        'category': Path(image_path).parent.name,
        'filename': Path(image_path).name,
    }

In [ ]:
def draw_detections(img_np, boxes, labels, scores, class_names, color=(0, 255, 0)):
    """Draw bounding boxes and labels onto a numpy image copy."""
    img = img_np.copy()
    for box, label, score in zip(boxes, labels, scores):
        x1, y1, x2, y2 = box.astype(int)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        if label < len(class_names):
            cls_name = class_names[label]
        else:
            cls_name = str(label)
        text = f'{cls_name} {score:.2f}'
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        cv2.rectangle(img, (x1, max(y1-th-4, 0)), (x1+tw+2, max(y1, th+4)), color, -1)
        cv2.putText(img, text, (x1+1, max(y1-3, th+1)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 1, cv2.LINE_AA)
    return img


def show_comparison(ssd_res, yolo_res, ssd_class_names, yolo_class_names, save=True):
    """Display SSD and YOLOv8 predictions side by side."""
    ssd_ann  = draw_detections(ssd_res['image'],  ssd_res['boxes'],  ssd_res['labels'],
                                ssd_res['scores'],  ssd_class_names,  color=(255, 80, 80))
    yolo_ann = draw_detections(yolo_res['image'], yolo_res['boxes'], yolo_res['labels'],
                                yolo_res['scores'], yolo_class_names, color=(80, 220, 80))

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    axes[0].imshow(cv2.cvtColor(ssd_ann,  cv2.COLOR_BGR2RGB) if ssd_ann.shape[2] == 3 else ssd_ann)
    axes[0].set_title(
        f'SSD300-VGG16 — {ssd_res["n_det"]} detections\nLatency: {ssd_res["latency_ms"]:.1f} ms',
        fontsize=12, color='#cc2200')
    axes[0].axis('off')

    axes[1].imshow(cv2.cvtColor(yolo_ann, cv2.COLOR_BGR2RGB) if yolo_ann.shape[2] == 3 else yolo_ann)
    axes[1].set_title(
        f'YOLOv8n — {yolo_res["n_det"]} detections\nLatency: {yolo_res["latency_ms"]:.1f} ms',
        fontsize=12, color='#00882a')
    axes[1].axis('off')

    category = ssd_res['category'].replace('_', ' ').title()
    plt.suptitle(f'[{category}]  {ssd_res["filename"]}', fontsize=13, fontweight='bold')
    plt.tight_layout()

    if save:
        out_path = f'comparison_{ssd_res["category"]}_{Path(ssd_res["filename"]).stem}.png'
        plt.savefig(out_path, dpi=120, bbox_inches='tight')
        print(f'  Saved → {out_path}')
    plt.show()

## Section 6 — Run the Investigation

All images in `stress_test_dataset/` are processed by both models. Results are collected into a list of dicts for building the evaluation matrix.

**Latency measurement:** `time.perf_counter()` wraps only the model forward pass — preprocessing (image loading, tensor conversion) and postprocessing (NMS) are included in the Ultralytics call but excluded from the torchvision SSD timing for fairness. Both include at least the backbone + head forward pass.

**Warmup:** One warmup inference is run before timing to eliminate CUDA/MPS initialization overhead from measurements.

In [ ]:
# Warmup pass (eliminate GPU/MPS init overhead)
if all_images:
    _ = run_ssd(all_images[0], ssd_model, DEVICE, CONF_THRESH)
    _ = run_yolo(all_images[0], yolo_model, CONF_THRESH, IOU_THRESH, IMG_SIZE)
    print('Warmup complete.')

ssd_results  = []
yolo_results = []

print(f'\nRunning inference on {len(all_images)} images...\n')
for img_path in all_images:
    ssd_r  = run_ssd(img_path,  ssd_model,  DEVICE, CONF_THRESH)
    yolo_r = run_yolo(img_path, yolo_model, CONF_THRESH, IOU_THRESH, IMG_SIZE)

    ssd_results.append(ssd_r)
    yolo_results.append(yolo_r)

    print(f'  [{img_path.parent.name}] {img_path.name}')
    print(f'    SSD  : {ssd_r["n_det"]:2d} det | {ssd_r["latency_ms"]:7.1f} ms')
    print(f'    YOLO : {yolo_r["n_det"]:2d} det | {yolo_r["latency_ms"]:7.1f} ms')

In [ ]:
# Display side-by-side comparison for each image
for ssd_r, yolo_r in zip(ssd_results, yolo_results):
    show_comparison(
        ssd_r, yolo_r,
        ssd_class_names=SSD_CLASSES_ORDERED,
        yolo_class_names=YOLO_CLASSES,
        save=True
    )

## Section 7 — Evaluation Matrix

The assignment requires documenting observations across three evaluation dimensions:

| Evaluation Metric | What to Measure |
|-------------------|-----------------|
| **Small Object Detection** | Number of missed objects in `small_distant/` and dense `crowded_scenes/` |
| **Temporal Efficiency** | Inference latency in ms/image; derived FPS |
| **Localization Precision** | Qualitative assessment of bounding box tightness and alignment |

### Localization Precision Scoring

Score each model 1–5 per image:

| Score | Description |
|-------|-------------|
| 5 | Boxes tightly fit object boundaries (IoU ≥ 0.8 with ground truth estimate) |
| 4 | Slight over/under coverage but object clearly enclosed |
| 3 | Moderate misalignment or box drift |
| 2 | Box loosely covers object; significant slack or offset |
| 1 | Box barely covers object or is a false positive |

In [ ]:
rows = []
for ssd_r, yolo_r in zip(ssd_results, yolo_results):
    rows.append({
        'Category':        ssd_r['category'],
        'Image':           ssd_r['filename'],
        'SSD Detections':  ssd_r['n_det'],
        'YOLO Detections': yolo_r['n_det'],
        'SSD Latency(ms)': round(ssd_r['latency_ms'], 1),
        'YOLO Latency(ms)':round(yolo_r['latency_ms'], 1),
        # Localization score is filled manually after visual inspection
        'SSD Localiz.':    '—',
        'YOLO Localiz.':   '—',
    })

eval_df = pd.DataFrame(rows)
print('\n=== FULL EVALUATION MATRIX ===')
print(eval_df.to_string(index=False))

print('\n=== AGGREGATE SUMMARY ===')
print(f'{"Model":<10} {"Avg Detections":>16} {"Avg Latency(ms)":>16} {"Approx FPS":>12}')
print('-' * 58)
ssd_avg_det  = eval_df['SSD Detections'].mean()
yolo_avg_det = eval_df['YOLO Detections'].mean()
ssd_avg_lat  = eval_df['SSD Latency(ms)'].mean()
yolo_avg_lat = eval_df['YOLO Latency(ms)'].mean()
print(f'{"SSD300":<10} {ssd_avg_det:>16.1f} {ssd_avg_lat:>16.1f} {1000/ssd_avg_lat:>12.1f}')
print(f'{"YOLOv8n":<10} {yolo_avg_det:>16.1f} {yolo_avg_lat:>16.1f} {1000/yolo_avg_lat:>12.1f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Chart 1: Detection count per image
x = np.arange(len(eval_df))
w = 0.35
axes[0].bar(x - w/2, eval_df['SSD Detections'],  w, label='SSD300',  color='#E05050', alpha=0.85)
axes[0].bar(x + w/2, eval_df['YOLO Detections'], w, label='YOLOv8n', color='#50C050', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"{r['Category'][:8]}\n{r['Image'][:8]}" for _, r in eval_df.iterrows()],
                         fontsize=7, rotation=30)
axes[0].set_title('Detections per Image')
axes[0].set_ylabel('# Detections')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.4)

# Chart 2: Latency per image
axes[1].bar(x - w/2, eval_df['SSD Latency(ms)'],  w, label='SSD300',  color='#E05050', alpha=0.85)
axes[1].bar(x + w/2, eval_df['YOLO Latency(ms)'], w, label='YOLOv8n', color='#50C050', alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"{r['Category'][:8]}\n{r['Image'][:8]}" for _, r in eval_df.iterrows()],
                         fontsize=7, rotation=30)
axes[1].set_title('Inference Latency (ms/image)')
axes[1].set_ylabel('Latency (ms)')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.4)

# Chart 3: Average summary bar
metrics = ['Avg Detections', 'Avg Latency (ms)', 'Approx FPS']
ssd_vals  = [ssd_avg_det,  ssd_avg_lat,  1000/ssd_avg_lat]
yolo_vals = [yolo_avg_det, yolo_avg_lat, 1000/yolo_avg_lat]
xm = np.arange(len(metrics))
axes[2].bar(xm - w/2, ssd_vals,  w, label='SSD300',  color='#E05050', alpha=0.85)
axes[2].bar(xm + w/2, yolo_vals, w, label='YOLOv8n', color='#50C050', alpha=0.85)
axes[2].set_xticks(xm)
axes[2].set_xticklabels(metrics, rotation=12)
axes[2].set_title('Aggregate Performance')
axes[2].legend()
axes[2].grid(axis='y', alpha=0.4)

plt.suptitle('SSD300 vs. YOLOv8n — Stress-Test Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('evaluation_summary.png', dpi=150)
plt.show()
print('Chart saved → evaluation_summary.png')

## Section 8 — Architectural Deep-Dive: The "Why"

---

### Question 5.1 — Spatial Constraints (Grid-Based Prediction)

**Context:** Early YOLO architectures (v1, v2) divided the input image into an S×S spatial grid. Each grid cell was responsible for predicting a fixed number B of bounding boxes and their class probabilities.

**Question:** *If two small objects occupy the same grid cell, what inherent prediction limitation does the model encounter?*

**Answer:**

In YOLOv1/v2, each grid cell can predict **at most B bounding boxes** (B=2 in v1), but all B boxes in a cell share the same set of class probability predictions. This creates two problems when two objects fall in the same grid cell:

1. **One object is dropped (spatial collision):** Because class probabilities are shared per-cell — not per-box — the cell can only commit to one dominant class. If a `cat` and a `dog` both have their centers in the same 7×7 cell, the model selects the class with the higher probability for that cell and effectively ignores the other object.

2. **Regression conflict:** Both boxes within the same cell compete to regress to different ground-truth objects. The gradient signal from two different targets in the same cell creates conflicting updates for the shared convolutional features, degrading both predictions.

**This is the "multi-object per cell" limitation** — a fundamental constraint of the grid-based assignment strategy. It particularly hurts detection of:
- Small objects clustered together (birds, pedestrians in a crowd)
- Objects at the same depth plane (cars in a parking lot viewed from above)

Modern YOLO versions (v3+) partially address this by predicting at **3 spatial scales** (13×13, 26×26, 52×52), reducing the probability of two objects sharing a cell. YOLOv8 goes further by removing anchors entirely — each object is assigned to its single best grid cell based on task-aligned scoring, and the decoupled head avoids the class-sharing limitation.

---

### Question 5.2 — Resolution & Feature Maps

**Context:** SSD performs detection using **6 feature maps at different spatial resolutions** (38×38, 19×19, 10×10, 5×5, 3×3, 1×1).

**Question:** *How does this multi-scale prediction strategy help SSD detect small objects more effectively than earlier grid-based YOLO architectures?*

**Answer:**

Small objects leave a very sparse activation footprint in deep feature maps. In YOLOv1, detection only happens at the final 7×7 feature map — by that point, the receptive field of each cell covers ~64 pixels of the original 448×448 input, and small objects (e.g., a 20×20 bird) activate only 0.1% of the final feature map. Their spatial signal is effectively averaged away.

SSD's multi-scale approach addresses this through **two complementary mechanisms:**

1. **High-resolution early layers for small objects:** The 38×38 Conv4_3 feature map (stride 8) has cells with smaller receptive fields (~72px). A small object that occupies only a few pixels in the final layer still activates a meaningful region at this resolution. Default boxes at the 38×38 level use scales of ~21–45px, specifically designed for small object detection.

2. **Appropriate anchor scales per level:** Each of the 6 detection heads uses default boxes with scales matched to its receptive field. Large objects (trucks, buses) are detected at 1×1 or 3×3 feature maps (very large receptive fields); small objects (birds, bottles) at 38×38 or 19×19 maps. This explicit scale assignment reduces the anchor-object size mismatch that caused YOLOv1 to miss small objects.

The trade-off: using early-layer feature maps for small object detection means those features are **semantically weaker** (less context about what the object is). SSD partially compensates with hard negative mining but still struggles with very small objects vs. FPN-based architectures that use a top-down pathway to inject semantics into early layers.

---

### Question 5.3 — The Unified Regression Paradigm

**Context:** YOLO treats object detection as a **unified regression problem** — directly predicting bounding box coordinates and class probabilities in a single network pass.

**Question:** *How does eliminating a separate "Region Proposal" stage improve inference speed and enable real-time detection?*

**Answer:**

Two-stage detectors (R-CNN, Fast R-CNN, Faster R-CNN) decompose detection into:
1. **Stage 1 — Region Proposal Network (RPN):** Scans the feature map and generates ~2,000 object candidate regions.
2. **Stage 2 — RoI Head:** Each proposal is individually pooled, classified, and box-regressed.

This pipeline introduces three latency penalties:
- **Sequential bottleneck:** Stage 2 cannot begin until Stage 1 completes — no parallelism across stages.
- **Per-proposal overhead:** 2,000 RoI pooling + classification operations per image adds significant compute.
- **Memory bandwidth:** Storing and retrieving feature crops for each proposal is memory-bound.

YOLO's unified regression eliminates all three:

1. **Single forward pass:** The backbone, neck, and detection head are executed once. All bounding boxes are predicted simultaneously from the feature grid — no iterative proposal generation.

2. **Fixed compute graph:** The number of operations is constant per image (determined by grid size, not object count). Faster R-CNN's compute scales with proposal count; YOLO's does not.

3. **No proposal filtering:** YOLO applies NMS once at the end on a fixed set of candidate locations. There is no intermediate filtering step that must be completed before regression can begin.

**Quantitative impact:** Faster R-CNN achieves ~5–7 FPS on a Titan X (2015). YOLOv1 achieved **45 FPS** on the same hardware — a ~9× speedup. YOLOv8n reaches **80–160 FPS** on modern GPUs, enabling deployment on embedded systems and real-time video at ≥30 FPS with full accuracy.

## Section 9 — Technical Synthesis

### Which architecture is better for a real-time traffic monitoring system?

**Recommendation: YOLOv8n (or larger YOLO variant)**

Traffic monitoring demands real-time throughput (≥25 FPS) across diverse object sizes — pedestrians close to the camera, vehicles at mid-range, and cyclists/motorcycles at distance. YOLOv8 satisfies all three requirements. Its anchor-free decoupled head achieves superior localization precision for vehicles, while its three-scale PAN-FPN neck handles both nearby pedestrians (small receptive field levels) and distant vehicles (large receptive field levels). Empirically, YOLOv8n's inference latency (~10–15 ms/frame on a modern GPU) comfortably exceeds the 30 FPS threshold, whereas SSD300's VGG-16 backbone introduces significantly higher latency (~30–80 ms depending on hardware). For embedded deployment (dashcams, roadside cameras), YOLOv8n's ~3M parameter count enables edge inference without a dedicated GPU.

### Which architecture is better for high-precision medical imaging?

**Recommendation: Mask R-CNN / Faster R-CNN (two-stage) — not SSD or YOLOv8 alone**

Medical imaging (tumor detection, polyp segmentation) prioritizes recall and localization precision over throughput. False negatives (missed lesions) carry life-critical consequences. Two-stage detectors with RoIAlign achieve higher mAP on small, ambiguous structures because the per-proposal refinement step corrects initial anchor misalignment. If forced to choose between SSD and YOLOv8 for this task, **YOLOv8 (medium/large variant with high-resolution input)** is preferred — its CIoU loss and decoupled head produce tighter bounding boxes than SSD's coupled regression. However, both are suboptimal vs. purpose-built medical segmentation architectures (U-Net, nnU-Net) that preserve full spatial resolution throughout the network.

## References

### Primary Papers

1. **SSD:** Liu, W., Anguelov, D., Erhan, D., et al. (2016). *SSD: Single Shot MultiBox Detector*. ECCV 2016.
2. **YOLO v1:** Redmon, J., Divvala, S., Girshick, R., & Farhadi, A. (2016). *You Only Look Once: Unified, Real-Time Object Detection*. CVPR 2016.
3. **YOLOv8:** Jocher, G., Chaurasia, A., & Qiu, J. (2023). *Ultralytics YOLOv8*. GitHub: ultralytics/ultralytics.
4. **VGG-16:** Simonyan, K. & Zisserman, A. (2015). *Very Deep Convolutional Networks for Large-Scale Image Recognition*. ICLR 2015.
5. **Faster R-CNN:** Ren, S., He, K., Girshick, R., & Sun, J. (2015). *Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks*. NeurIPS 2015.
6. **COCO Dataset:** Lin, T. Y., et al. (2014). *Microsoft COCO: Common Objects in Context*. ECCV 2014.

### Configuration Reference

| Parameter | Value | Source |
|-----------|-------|--------|
| Confidence Threshold | 0.25 | Assignment specification |
| IoU Threshold (NMS) | 0.45 | Assignment specification |
| Input Resolution | 640×640 | Assignment specification |
| SSD internal resolution | 300×300 | SSD300 architecture constraint |
| YOLOv8n pretrained | COCO 2017 | Ultralytics default checkpoint |
| SSD300 pretrained | COCO 2017 | torchvision default weights |
| Device priority | CUDA → MPS → CPU | Auto-detected at runtime |